In [7]:
import re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd


In [27]:
def decode_vnnlib_to_image_and_label(
    vnnlib_path: str,
    *,
    cifar_layout: str = "CHW",     # most common for CIFAR: CHW
    mean=None,                    # optional denorm
    std=None,                     # optional denorm
):
    """
    Returns:
      img_chw_or_hw (float32): reconstructed image (CHW if CIFAR and layout=CHW, else HWC or 2D)
      label (int|None): inferred label (from output constraints)
    """
    vnnlib_path = Path(vnnlib_path)
    text = vnnlib_path.read_text(errors="ignore")

    # Parse bounds for X_i
    re_le = re.compile(r"\(assert\s*\(\s*<=\s*X_(\d+)\s*([+-]?(?:\d+\.?\d*|\.\d+)(?:[eE][+-]?\d+)?)\s*\)\s*\)")
    re_ge = re.compile(r"\(assert\s*\(\s*>=\s*X_(\d+)\s*([+-]?(?:\d+\.?\d*|\.\d+)(?:[eE][+-]?\d+)?)\s*\)\s*\)")

    lb, ub = {}, {}

    for m in re_ge.finditer(text):
        i = int(m.group(1)); v = float(m.group(2))
        lb[i] = max(lb.get(i, -np.inf), v)

    for m in re_le.finditer(text):
        i = int(m.group(1)); v = float(m.group(2))
        ub[i] = min(ub.get(i, np.inf), v)

    if not lb and not ub:
        raise ValueError("No X_i bounds found.")

    all_idx = sorted(set(lb.keys()) | set(ub.keys()))
    n = all_idx[-1] + 1

    lb_arr = np.full((n,), -np.inf, dtype=np.float32)
    ub_arr = np.full((n,),  np.inf, dtype=np.float32)

    for i, v in lb.items(): lb_arr[i] = v
    for i, v in ub.items(): ub_arr[i] = v

    # Fill one-sided bounds
    only_lb = (~np.isinf(lb_arr)) & (np.isinf(ub_arr))
    only_ub = (np.isinf(lb_arr)) & (~np.isinf(ub_arr))
    both_missing = (np.isinf(lb_arr)) & (np.isinf(ub_arr))

    ub_arr[only_lb] = lb_arr[only_lb]
    lb_arr[only_ub] = ub_arr[only_ub]
    lb_arr[both_missing] = 0.0
    ub_arr[both_missing] = 0.0

    x_mid = (lb_arr + ub_arr) / 2.0

    # Infer label from Y comparisons (>= Y_a Y_b) -> pick most frequent RHS b
    re_ycomp = re.compile(r"\(>=\s*Y_(\d+)\s+Y_(\d+)\)")
    rhs_counts = {}
    for m in re_ycomp.finditer(text):
        rhs = int(m.group(2))
        rhs_counts[rhs] = rhs_counts.get(rhs, 0) + 1

    label = max(rhs_counts.items(), key=lambda kv: kv[1])[0] if rhs_counts else None

    # Reshape
    if n == 3072:
        if cifar_layout.upper() == "CHW":
            img = x_mid.reshape(3, 32, 32).astype(np.float32)  # CHW
        else:
            img = x_mid.reshape(32, 32, 3).astype(np.float32)  # HWC
    else:
        side = int(round(np.sqrt(n)))
        if side * side == n:
            img = x_mid.reshape(side, side).astype(np.float32)
        else:
            img = x_mid.astype(np.float32)  # flat fallback

    # Optional denorm (if image has channels)
    if mean is not None and std is not None and img.ndim == 3:
        mean = np.array(mean, dtype=np.float32)
        std = np.array(std, dtype=np.float32)

        if img.shape[0] == 3:  # CHW
            img = (img * std[:, None, None]) + mean[:, None, None]
        elif img.shape[-1] == 3:  # HWC
            img = (img * std[None, None, :]) + mean[None, None, :]

    return img, label


def batch_decode_vnnlib_folder_to_png(
    vnnlib_dir: str,
    out_dir: str,
    *,
    recursive: bool = True,
    cifar_layout: str = "CHW",
    mean=None,
    std=None,
):
    vnnlib_dir = Path(vnnlib_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(vnnlib_dir.rglob("*.vnnlib") if recursive else vnnlib_dir.glob("*.vnnlib"))
    if not files:
        raise FileNotFoundError(f"No .vnnlib files found in: {vnnlib_dir}")

    # try to parse sidx from filename like ..._sidx_313_...
    re_sidx = re.compile(r"_sidx_(\d+)", re.I)

    rows = []
    for i, vp in enumerate(files):
        try:
            img, label = decode_vnnlib_to_image_and_label(
                str(vp),
                cifar_layout=cifar_layout,
                mean=mean,
                std=std,
            )

            m = re_sidx.search(vp.name)
            sidx = int(m.group(1)) if m else None
            m_model = re.search(r"CIFAR100_([^_]+(?:_[^_]+)*)_prop_idx", vp.name)
            model = m_model.group(1) if m_model else "unknown_model"

            safe_label = "unknown" if label is None else str(label)

            if sidx is not None:
                fname = f"{model}__label_{safe_label}__idx_{sidx}.png"
            else:
                fname = f"{model}__label_{safe_label}__i_{i}.png"

            # Convert to HWC for saving
            if img.ndim == 3 and img.shape[0] == 3:  # CHW -> HWC
                img_hwc = np.transpose(img, (1, 2, 0))
            else:
                img_hwc = img

            # Save PNG (clip to [0,1] for viewability)
            if img_hwc.ndim == 3:
                u8 = (np.clip(img_hwc, 0.0, 1.0) * 255.0 + 0.5).astype(np.uint8)
                pil = Image.fromarray(u8, mode="RGB")
            elif img_hwc.ndim == 2:
                u8 = (np.clip(img_hwc, 0.0, 1.0) * 255.0 + 0.5).astype(np.uint8)
                pil = Image.fromarray(u8, mode="L")
            else:
                # flat fallback: can't save meaningful PNG; skip
                raise ValueError(f"Flat input (n={img_hwc.shape[0]}) cannot be saved as an image without shape info.")

            out_path = out_dir / fname
            pil.save(out_path)

            rows.append({
                "vnnlib": str(vp),
                "label": label,
                "sidx": sidx,
                "png": str(out_path),
                "status": "ok",
            })
            print(f"[{i+1:4d}/{len(files)}] {vp.name} -> {out_path.name}")

        except Exception as e:
            rows.append({
                "vnnlib": str(vp),
                "label": None,
                "sidx": None,
                "png": None,
                "status": "error",
                "error": repr(e),
            })
            print(f"[{i+1:4d}/{len(files)}] {vp.name} -> {e}")

    df = pd.DataFrame(rows)
    df.to_csv(out_dir / "decode_png_summary.csv", index=False)
    return df

In [29]:
VNNLIB_DIR = "vnnlib"   # <-- change this
OUT_DIR   = "decoded_vnnlibs"                  # <-- output folder

df = batch_decode_vnnlib_folder_to_png(
    VNNLIB_DIR,
    OUT_DIR,
    recursive=True,
    cifar_layout="CHW",
    mean=(0.5071, 0.4867, 0.4408),
    std=(0.2675, 0.2565, 0.2761),
)

df.head()

[   1/200] CIFAR100_resnet_large_prop_idx_1059_sidx_6596_eps_0.0039.vnnlib -> resnet_large__label_49__idx_6596.png
[   2/200] CIFAR100_resnet_large_prop_idx_1063_sidx_7948_eps_0.0039.vnnlib -> resnet_large__label_83__idx_7948.png
[   3/200] CIFAR100_resnet_large_prop_idx_1105_sidx_9000_eps_0.0039.vnnlib -> resnet_large__label_94__idx_9000.png
[   4/200] CIFAR100_resnet_large_prop_idx_1216_sidx_6431_eps_0.0039.vnnlib -> resnet_large__label_48__idx_6431.png
[   5/200] CIFAR100_resnet_large_prop_idx_1241_sidx_5679_eps_0.0039.vnnlib -> resnet_large__label_86__idx_5679.png
[   6/200] CIFAR100_resnet_large_prop_idx_1280_sidx_5310_eps_0.0039.vnnlib -> resnet_large__label_89__idx_5310.png
[   7/200] CIFAR100_resnet_large_prop_idx_1298_sidx_33_eps_0.0039.vnnlib -> resnet_large__label_76__idx_33.png
[   8/200] CIFAR100_resnet_large_prop_idx_1309_sidx_8525_eps_0.0039.vnnlib -> resnet_large__label_12__idx_8525.png
[   9/200] CIFAR100_resnet_large_prop_idx_1426_sidx_3855_eps_0.0039.vnnlib -> resnet

,vnnlib,label,sidx,png,status
0,vnnlib/CIFAR100_resnet_large_prop_idx_1059_sid...,49,6596,decoded_vnnlibs/resnet_large__label_49__idx_65...,ok
1,vnnlib/CIFAR100_resnet_large_prop_idx_1063_sid...,83,7948,decoded_vnnlibs/resnet_large__label_83__idx_79...,ok
2,vnnlib/CIFAR100_resnet_large_prop_idx_1105_sid...,94,9000,decoded_vnnlibs/resnet_large__label_94__idx_90...,ok
3,vnnlib/CIFAR100_resnet_large_prop_idx_1216_sid...,48,6431,decoded_vnnlibs/resnet_large__label_48__idx_64...,ok
4,vnnlib/CIFAR100_resnet_large_prop_idx_1241_sid...,86,5679,decoded_vnnlibs/resnet_large__label_86__idx_56...,ok


-------

In [12]:
import os
import sys
from pathlib import Path
import random
import numpy as np
from PIL import Image

# -----------------------------
# 1) User inputs
# -----------------------------
IMG_DIR = Path("decoded_vnnlibs")   # <-- change to your folder of reconstructed PNGs
N = None                         # None = all images, or set e.g. 200
FILL_MODE = "white"              # "white" or "black"
SAMPLE_MODE = "first"            # "first" or "random"

assert IMG_DIR.exists(), f"Image folder does not exist: {IMG_DIR}"
assert FILL_MODE in {"white", "black"}
assert SAMPLE_MODE in {"first", "random"}

# Output (single folder, like your notebook)
OUT_DIR  = IMG_DIR.parent / f"{IMG_DIR.name}_sam2_split"
OBJ_DIR  = OUT_DIR / "object"
BG_DIR   = OUT_DIR / "background"
MASK_DIR = OUT_DIR / "masks"   # optional (debug)

for d in [OBJ_DIR, BG_DIR, MASK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Saving results to:", OUT_DIR)

# Collect images
img_files = [p for p in IMG_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
img_files = sorted(img_files)

if SAMPLE_MODE == "random":
    random.shuffle(img_files)

if N is not None:
    img_files = img_files[:int(N)]

print(f"Found {len(img_files)} images to process.")

Saving results to: decoded_vnnlibs_sam2_split
Found 195 images to process.


In [15]:
# -----------------------------
# 2) Import your sam2_utils
# -----------------------------
# Set PROJECT_ROOT to the folder that contains `scripts/sam2_utils.py`
PROJECT_ROOT = Path("../../../..")   # <-- change if needed

assert PROJECT_ROOT.exists(), f"Project root does not exist: {PROJECT_ROOT}"
sys.path.insert(0, str(PROJECT_ROOT))

try:
    from scripts.sam2_utils import load_sam2_predictor, run_segmentation_model
    print("Imported: scripts.sam2_utils")
except Exception as e:
    raise ImportError(
        "Could not import scripts.sam2_utils.\n"
        "Make sure PROJECT_ROOT is correct and contains scripts/sam2_utils.py\n"
        f"Original error: {e}"
    )

Imported: scripts.sam2_utils


In [22]:
# -----------------------------
# 3) SAM2 config + predictor
# -----------------------------
SEG_CFG = {
    "model_name": "facebook/sam2-hiera-large",   # <-- match what your project expects
    "device": "cuda" if os.environ.get("CUDA_VISIBLE_DEVICES") not in (None, "", "-1") else "cpu",
}

print("SEG_CFG =", SEG_CFG)
predictor = load_sam2_predictor(SEG_CFG)
print("Loaded SAM2 predictor.")

SEG_CFG = {'model_name': 'facebook/sam2-hiera-large', 'device': 'cuda'}
[SAM2] Using device: cuda
Loaded SAM2 predictor.


In [23]:
import numpy as np

def _as_numpy(x):
    try:
        import torch
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().numpy()
    except Exception:
        pass
    return np.asarray(x)

def _to_binary_mask(mask):
    m = _as_numpy(mask)

    # handle (H,W), (1,H,W), (N,H,W), (H,W,1)
    if m.ndim == 3:
        if m.shape[0] == 1:
            m = m[0]
        elif m.shape[-1] == 1:
            m = m[..., 0]
        # if m is (N,H,W) keep it for caller to split
    if m.dtype != bool:
        m = m > 0.5
    return m.astype(bool)

def _score_to_float(s):
    """
    Convert score-like objects to float.
    Handles:
      - float/int
      - numpy scalar
      - dict like {"iou": 0.9, ...}
    """
    if s is None:
        return None
    if isinstance(s, (float, int, np.floating, np.integer)):
        return float(s)
    if isinstance(s, dict):
        # Try common keys
        for k in ["predicted_iou", "iou", "score", "confidence", "stability_score"]:
            v = s.get(k, None)
            if isinstance(v, (float, int, np.floating, np.integer)):
                return float(v)
        # fallback: first numeric value
        for v in s.values():
            if isinstance(v, (float, int, np.floating, np.integer)):
                return float(v)
        return None
    # fallback: try float()
    try:
        return float(s)
    except Exception:
        return None

def _extract_masks_and_scores(out):
    """
    Supports:
      - dict with keys: masks / segments / mask
      - list of segment dicts: [ {mask:.., score:..}, ... ]  (your case)
      - tuple/list (masks, scores)
      - raw numpy/torch mask
    Returns:
      masks_list: list of masks (numpy)
      scores_list: list of floats or None
    """

    # ---- CASE A: out is a list of segment dicts ----
    if isinstance(out, list) and len(out) > 0 and all(isinstance(x, dict) for x in out):
        masks_list = []
        scores_list = []
        for seg in out:
            m = seg.get("mask", seg.get("segmentation", None))
            if m is None:
                continue
            masks_list.append(_as_numpy(m))
            scores_list.append(_score_to_float(seg.get("score", seg.get("iou", seg.get("predicted_iou", None)))))
        return masks_list, (None if all(s is None for s in scores_list) else scores_list)

    # ---- CASE B: dict outputs ----
    if isinstance(out, dict):
        if "masks" in out:
            masks = _as_numpy(out["masks"])
            scores = out.get("scores", out.get("iou_scores", out.get("predicted_iou", None)))

            scores_list = None
            if scores is not None:
                scores_np = _as_numpy(scores).reshape(-1)
                scores_list = [_score_to_float(s) for s in scores_np.tolist()]

            if masks.ndim == 2:
                return [masks], scores_list
            if masks.ndim == 3:
                return [masks[i] for i in range(masks.shape[0])], scores_list
            raise ValueError(f"Unexpected masks shape: {masks.shape}")

        if "segments" in out:
            masks_list = []
            scores_list = []
            for seg in out["segments"]:
                if not isinstance(seg, dict):
                    continue
                m = seg.get("mask", seg.get("segmentation", None))
                if m is None:
                    continue
                masks_list.append(_as_numpy(m))
                scores_list.append(_score_to_float(seg.get("score", seg.get("iou", seg.get("predicted_iou", None)))))
            return masks_list, (None if all(s is None for s in scores_list) else scores_list)

        if "mask" in out:
            return [_as_numpy(out["mask"])], None

        raise ValueError(f"Unknown dict keys from run_segmentation_model: {list(out.keys())}")

    # ---- CASE C: tuple/list like (masks, scores) ----
    if isinstance(out, (tuple, list)) and len(out) >= 1:
        # If it's list but not list-of-dicts, treat first item as masks
        masks = _as_numpy(out[0])
        scores_list = None

        if len(out) >= 2:
            scores = out[1]
            if scores is not None:
                s_np = _as_numpy(scores).reshape(-1)
                scores_list = [_score_to_float(s) for s in s_np.tolist()]

        if masks.ndim == 2:
            return [masks], scores_list
        if masks.ndim == 3:
            return [masks[i] for i in range(masks.shape[0])], scores_list

    # ---- CASE D: raw mask ----
    masks = _as_numpy(out)
    if masks.ndim == 2:
        return [masks], None
    if masks.ndim == 3:
        return [masks[i] for i in range(masks.shape[0])], None

    raise ValueError(f"Unexpected output type/shape from run_segmentation_model: {type(out)} {getattr(out,'shape',None)}")

def segment_image_grid(img_pil: Image.Image, grid_n: int = 8):
    arr = np.array(img_pil.convert("RGB"))
    H, W = arr.shape[:2]
    pts, labs = make_grid_points(W, H, grid_n=grid_n)

    out = None
    try:
        out = run_segmentation_model(predictor, arr, points=pts, labels=labs)
    except TypeError:
        try:
            out = run_segmentation_model(predictor, arr, point_coords=pts, point_labels=labs)
        except TypeError:
            out = run_segmentation_model(predictor, arr)

    masks_list, scores = _extract_masks_and_scores(out)
    if not masks_list:
        raise ValueError("SAM2 returned no masks after parsing.")

    # ---- IMPORTANT: pick seg0 only ----
    m0 = _as_numpy(masks_list[0])

    # If seg0 is (N,H,W), pick the biggest slice by area (rare)
    if m0.ndim == 3:
        areas = [int(_to_binary_mask(m0[i]).sum()) for i in range(m0.shape[0])]
        m0 = m0[int(np.argmax(areas))]

    mask = _to_binary_mask(m0)

    # Ensure size matches
    if mask.shape != (H, W):
        raise ValueError(f"Mask shape {mask.shape} != image shape {(H,W)}")

    return mask

In [24]:
FILL = 255 if FILL_MODE == "white" else 0

ok = 0
bad = 0

for idx, p in enumerate(img_files, 1):
    try:
        img = Image.open(p).convert("RGB")
        arr = np.array(img)  # keep original size

        mask = segment_image_grid(img, grid_n=8)

        if mask.shape[:2] != arr.shape[:2]:
            raise ValueError(f"Mask shape {mask.shape} != image shape {arr.shape}")

        fill_bg = np.ones_like(arr, dtype=np.uint8) * FILL

        obj_img = np.where(mask[..., None], arr, fill_bg)
        bg_img  = np.where(~mask[..., None], arr, fill_bg)

        # Save with same filename (already label_idx.png)
        Image.fromarray(obj_img).save(OBJ_DIR / p.name)
        Image.fromarray(bg_img).save(BG_DIR / p.name)

        # Optional masks (debug)
        Image.fromarray((mask.astype(np.uint8) * 255)).save(MASK_DIR / p.name)

        ok += 1
        if idx % 25 == 0:
            print(f"[{idx}/{len(img_files)}] ✅ done (ok={ok}, bad={bad})")

    except Exception as e:
        bad += 1
        print(f"[{idx}/{len(img_files)}] ❌ {p.name} -> {e}")

print(f"Finished. ok={ok}, bad={bad}")
print("Object images:", OBJ_DIR)
print("Background images:", BG_DIR)
print("Masks:", MASK_DIR)

[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
[SAM2] Found 3 segments after filtering
